In [1]:
from openai import OpenAI
import os
import json
import subprocess
import gradio as gr

d:\programing\git projects\coder\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#llm init in agent
def model_init(api_key,base_url)->object:
    model=OpenAI(api_key=api_key,base_url=base_url)
    return model

In [3]:
#tools
def read_file(path)->str:
    with open(path,"r") as file:
        text=file.read()
    return text
    
def write_file(path,code)->None:
    with open(path,"w") as file:
        file.write(code)

def list_files(path=".")->list:
    return os.listdir(path)

def run_command(command)->dict:
    result= subprocess.run(
        command,
        shell=True,
        capture_output=True,
        text=True
    )
    return {
        "stdout": result.stdout,
        "stderr": result.stderr,
        "returncode": result.returncode
    }

In [4]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read the contents of a file.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {
                        "type": "string",
                        "description": "The path of the file to read."
                    }
                },
                "required": ["path"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "write_file",
            "description": "Write code or text to a file.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {
                        "type": "string",
                        "description": "The path of the file to write."
                    },
                    "code": {
                        "type": "string",
                        "description": "The code or text to write into the file."
                    }
                },
                "required": ["path", "code"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "list_files",
            "description": "List all files and folders inside a directory.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {
                        "type": "string",
                        "description": "The directory path to list. Defaults to the current directory."
                    }
                },
                "required": []
            }
        }
    },
    {
    "type": "function",
    "function": {
        "name": "run_command",
        "description": "Run a shell command and return its output.",
        "parameters": {
            "type": "object",
            "properties": {
                "command": {
                    "type": "string",
                    "description": "The shell command to execute."
                }
            },
            "required": ["command"]
        }
    }
  }
]

In [ ]:
def get_tool(tool_name,args):
    tool_dic={
        "read_file":read_file,
        "write_file":write_file,
        "list_files":list_files,
        "run_command":run_command
    }

    return tool_dic[tool_name](**args)

def tool_handler(msg):
    responses=[]
    for tool_call in msg.tool_calls:
        tool_name=tool_call.function.name
        args=json.loads(tool_call.function.arguments)
        result=get_tool(tool_name=tool_name,args=args)

        responses.append({
            "role":"tool",
            "content":json.dumps(result),
            "tool_call_id":tool_call.id
        })
    return responses

In [ ]:
def message(user_msg):
    sys_msg="""you are a coding agent.
    you modify the code according to the user request.
    before modifying a file, read the file first.
    use the available tools when you need to read and modify a file or when you need to test the code
    if an error accures you inspect the error and fix it.
    keep your words simple and precise"""

    msg=[{"role":"system","content":sys_msg},{"role":"user","content":user_msg}]
    return msg

In [ ]:
def coder(model,model_name,reasoning,msg):
    messages=message(msg)
    response=model.chat.completions.create(messages=messages,model=model_name,reasoning_effort=reasoning,tools=tools)

    while True:
        msg=response.choices[0].message
        if not msg.tool_calls:
            return msg.content
        messages.append(msg)
        tool_responses=tool_handler(msg)
        messages.extend(tool_responses)

        response=model.chat.completions.create(messages=messages,model=model_name,reasoning_effort=reasoning,tools=tools)

     

In [ ]:
def chat(message,history)->str:
    return message

def interface()->None:
    with gr.Blocks() as app:

        with gr.Row():

            with gr.Column(scale=1):
                gr.Markdown("Files")

                files=gr.FileExplorer(
                    glob="workspace/**/*",
                    root_dir=".",
                    file_count="multiple",
                    label="Project"
            )

            with gr.Column(scale=3):
                chatbot=gr.Chatbot(
                    label="agent",
                    type="messages"
            )

                message=gr.Textbox(
                    placeholder="type the file and the change you want in it",
            )

                send=gr.Button("send")

        send.click(
            chat,
            inputs=[message,chatbot],
            outputs=chatbot
        )

        message.submit(
            chat,
            inputs=[message,chatbot],
            outputs=chatbot
        )
    app.launch()